# FunnyBirds MCBM — Renderer-Based Part Swap (Gamma Sweep)

**Replicates `fb_cbm_renderer_swap_v2.ipynb` for MCBM across all trained γ values.**

**MCBM vs CBM key differences:**
- Checkpoint keys: `concept_encoder.weight/bias` (not `concept_head`)
- z is *raw* (no sigmoid): `z_raw = W_c @ avgpool + b_c ∈ ℝ`; label_head takes z_raw directly
- GT ceiling: z_raw = +3 (active) / −3 (inactive), matching IB q_phi saturation range
- γ = 0.0 → plain CBM (sanity check); higher γ → stronger IB → expect less backwash

**Gammas:** [0.0, 0.1, 0.5, 1.0, 5.0]

In [ ]:
import gc
import io
import json
import os
import random
import subprocess
import sys
import time
from base64 import decodebytes
from itertools import combinations
from pathlib import Path

import requests
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats
import torch
import torch.nn as nn
from torchvision import models, transforms
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

## 0. Paths and configuration

In [ ]:
ROOT = Path('/scratch/network/cr7998/cv_emergence_project')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FB           = ROOT / 'data' / 'FunnyBirds'
RENDERER_DIR = ROOT.parent / 'funnybirds' / 'render'

GAMMAS = [0.0, 0.1, 0.5, 1.0, 5.0]

def mcbm_ckpt(g):  return ROOT / 'checkpoints_funnybirds' / f'mcbm_fb_gamma{g}.pth'
def mcbm_feats(g): return ROOT / 'features' / f'resnet50_mcbm_fb_gamma{g}'

assert FB.exists(), f'Missing FunnyBirds data: {FB}'
for g in GAMMAS:
    c, f = mcbm_ckpt(g), mcbm_feats(g)
    print(f'  gamma={g}  ckpt={c.exists()}  feats={f.exists()}')

N_SPECIES  = 50
N_CONCEPTS = 26

MAX_IMGS_PER_SPECIES = 5
MAX_PAIRS_PER_PART   = 100

# GT ceiling constants for MCBM (IB q_phi saturation range is [-3, 3])
MCBM_Z_ACTIVE   =  3.0
MCBM_Z_INACTIVE = -3.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
print(f'MAX_IMGS_PER_SPECIES={MAX_IMGS_PER_SPECIES}  MAX_PAIRS_PER_PART={MAX_PAIRS_PER_PART}')

## 1. Data loading helpers  *(from fb_recallv2.py)*

In [ ]:
def load_species_maps(fb_root: Path):
    classes_csv = fb_root / 'metadata' / 'classes.csv'
    if not classes_csv.exists():
        raise FileNotFoundError('metadata/classes.csv not found. Run prepare_funnybirds_metadata.py first.')
    df = pd.read_csv(classes_csv)
    id2name  = dict(zip(df['class_id'], df['class_name']))
    id2short = {k: v.replace('funnybird_', 'FB') for k, v in id2name.items()}
    return id2name, id2short


def load_meta(fb_root: Path) -> pd.DataFrame:
    images_csv = fb_root / 'metadata' / 'images.csv'
    if not images_csv.exists():
        raise FileNotFoundError('metadata/images.csv not found.')
    df = pd.read_csv(images_csv)
    id2name, _ = load_species_maps(fb_root)
    df['species_id']   = df['class_id']
    df['species_name'] = df['class_id'].map(id2name)
    return df


def safe_torch_load(path: Path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def load_features(feat_dir: Path, layer: str, split: str) -> torch.Tensor:
    p = feat_dir / f'{layer}_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    X = safe_torch_load(p)
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X)
    return X.float()


def to_1d_int_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.array(x).reshape(-1).astype(int)


def load_split_order(feat_dir: Path, split: str):
    p = feat_dir / f'labels_{split}.pt'
    assert p.exists(), f'Missing: {p}'
    t = safe_torch_load(p)
    assert isinstance(t, dict), f'Expected dict in {p}'
    assert 'image_ids' in t, f'{p} missing image_ids'
    ids = to_1d_int_array(t['image_ids'])
    return ids

print('Defined: data loading helpers')

## 2. Concept names, class-concept matrix, metadata

In [ ]:
from datasets.funnybirds_dataset import FunnyBirdsDataset
from datasets.funnybirds_dataset import concept_names as _fb_concept_names
from datasets.funnybirds_dataset import _build_part_lookup, _params_to_variant_idx
from datasets.funnybirds_dataset import PART_VARIANTS, _FUNNYBIRDS_N_TRAIN

CONCEPT_NAMES  = _fb_concept_names()   # 26 strings: ['beak_0', ..., 'tail_8']
CONCEPT_TO_IDX = {c: i for i, c in enumerate(CONCEPT_NAMES)}

PARTS = ['beak', 'eye', 'wing', 'foot', 'tail']
PART_GROUPS = {
    part: [i for i, c in enumerate(CONCEPT_NAMES) if c.startswith(f'{part}_')]
    for part in PARTS
}
PART_COLORS = {
    'beak': 'steelblue', 'eye': 'purple', 'wing': 'seagreen',
    'foot': 'darkorange', 'tail': 'crimson',
}
CONCEPT_TO_PART = {
    c: part
    for part, idxs in PART_GROUPS.items()
    for c in [CONCEPT_NAMES[i] for i in idxs]
}

_fb_ds    = FunnyBirdsDataset(FB, split='train')
cc_matrix, _ = _fb_ds.get_class_concept_matrix()   # [50, 26]
cc_matrix = cc_matrix.numpy()

print(f'Concepts ({len(CONCEPT_NAMES)}): {CONCEPT_NAMES}')
print(f'CC matrix: {cc_matrix.shape}')

In [ ]:
meta       = load_meta(FB)
id2name, _ = load_species_maps(FB)
def spname(sid: int) -> str:
    return id2name.get(int(sid), f'funnybird_{int(sid):02d}')

_meta_idx = meta.set_index('image_id')
print(f'meta: {len(meta)} images  ({meta["is_train"].sum()} train)')

## 3. FunnyBirds rendering parameters

In [ ]:
ann_path_test = FB / 'dataset_test.json'
assert ann_path_test.exists(), f'{ann_path_test} not found.'

with open(ann_path_test) as f:
    test_anns = json.load(f)

N_TEST = len(test_anns)
print(f'Loaded {N_TEST} test annotations')
print(f'Fields: {list(test_anns[0].keys())}')

In [ ]:
parts_path = FB / 'parts.json'
assert parts_path.exists(), f'parts.json not found at {parts_path}'

with open(parts_path) as f:
    parts_json = json.load(f)

parts_lookup = _build_part_lookup(parts_json)

PARTS_WITH_COLOR = set()
for part, variants in parts_json.items():
    if any('color' in v for v in variants):
        PARTS_WITH_COLOR.add(part)

print(f'Parts with color field: {PARTS_WITH_COLOR}')


def variant_idx_from_ann(ann: dict, part: str) -> int:
    model = ann.get(f'{part}_model', '')
    if not model or model == 'placeholder':
        return -1
    key_fields = {'model': model}
    if part in PARTS_WITH_COLOR:
        color = ann.get(f'{part}_color', '')
        if color:
            key_fields['color'] = color
    key = tuple(sorted(key_fields.items()))
    return parts_lookup[part].get(key, -1)


species_part_params = {}
species_variant_idx = {}

for ann in test_anns:
    sid = int(ann['class_idx'])
    if sid in species_part_params:
        continue
    species_part_params[sid] = {}
    species_variant_idx[sid] = {}
    for part in PARTS:
        params = {'model': ann.get(f'{part}_model', '')}
        if part in PARTS_WITH_COLOR:
            params['color'] = ann.get(f'{part}_color', '')
        species_part_params[sid][part]  = params
        species_variant_idx[sid][part]  = variant_idx_from_ann(ann, part)

print(f'Extracted part params for {len(species_part_params)} species')

# Per-species test image indices
test_idx_by_species = {sid: [] for sid in range(N_SPECIES)}
for local_idx, ann in enumerate(test_anns):
    sid = int(ann['class_idx'])
    test_idx_by_species[sid].append(local_idx)

print(f'Tail variant counts:')
tail_var_to_sids = {}
for sid in range(N_SPECIES):
    v = species_variant_idx[sid]['tail']
    tail_var_to_sids.setdefault(v, []).append(sid)
for v, sids in sorted(tail_var_to_sids.items()):
    print(f'  tail_{v}: {len(sids)} species')

## 4. Renderer setup

In [ ]:
def json_to_url(sample: dict,
                prefix: str = 'http://localhost:8081/render?',
                render_mode: str = 'default') -> str:
    url = prefix + 'render_mode=' + render_mode + '&'
    for key in list(sample.keys()):
        if key == 'class_idx':
            continue
        url = url + key + '=' + str(sample[key]) + '&'
    return url[:-1]


def json_to_image(sample: dict, mode: str = 'test') -> Image.Image:
    if mode in ('train', 'test'):
        url = json_to_url(sample)
    elif mode in ('train_part_map', 'test_part_map'):
        url = json_to_url(sample, render_mode='part_map')
    else:
        raise NotImplementedError(f'Unknown mode: {mode}')
    response = requests.get(url, timeout=30).content
    img_bytes = decodebytes(response)
    img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
    resample = Image.NEAREST if 'part_map' in mode else Image.BILINEAR
    return img.resize((256, 256), resample=resample)


_server_proc = None

def render_ann_safe(ann: dict, max_retries: int = 3) -> Image.Image:
    for attempt in range(max_retries):
        try:
            return json_to_image(ann, mode='test')
        except Exception:
            if attempt == max_retries - 1:
                raise
            try: _server_proc.kill()
            except: pass
            globals()['_server_proc'] = subprocess.Popen(
                ['node', 'server.js'], cwd=str(RENDERER_DIR),
                stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            time.sleep(6)

def render_ann(ann): return render_ann_safe(ann)
print('Defined: json_to_url  json_to_image  render_ann')

In [ ]:
def check_renderer_alive(timeout: float = 2.0) -> bool:
    try:
        r = requests.get(
            'http://localhost:8081/render?render_mode=default'
            '&beak_model=beak01.glb&eye_model=eye01.glb'
            '&foot_model=foot01.glb&tail_model=tail01.glb&tail_color=red'
            '&wing_model=wing01.glb&wing_color=red'
            '&camera_distance=300&camera_pitch=0&camera_roll=0'
            '&light_distance=300&light_pitch=0&light_roll=0',
            timeout=timeout,
        )
        return r.status_code == 200
    except Exception:
        return False


def start_renderer_server(renderer_dir: Path) -> bool:
    global _server_proc
    if check_renderer_alive():
        print('[renderer] Server already running on port 8081.')
        return True
    if not (renderer_dir / 'server.js').exists():
        print(f'[renderer] server.js not found at {renderer_dir}')
        return False
    print(f'[renderer] Starting server: node server.js in {renderer_dir} ...')
    _server_proc = subprocess.Popen(
        ['node', 'server.js'], cwd=str(renderer_dir),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    for _ in range(15):
        time.sleep(1)
        if check_renderer_alive():
            print('[renderer] Server is up.')
            return True
    print('[renderer] Server did not respond after 15 s.')
    return False


RENDERER_AVAILABLE = start_renderer_server(RENDERER_DIR)
print(f'RENDERER_AVAILABLE = {RENDERER_AVAILABLE}')

In [ ]:
eval_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print('eval_tf defined')

## 5. MCBM loading helpers

Key difference from CBM:
- Checkpoint key: `concept_encoder.weight/bias` (not `concept_head`)
- z_raw = W_c @ avgpool + b_c — **no sigmoid**; label_head takes z_raw directly
- For display: sigmoid(z_raw) gives concept "probability" proxy

In [ ]:
def load_mcbm_weights(ckpt_path):
    """
    Load MCBM checkpoint; return (W_c, b_c, W_y, b_y, config).
    W_c [26,2048], b_c [26], W_y [50,26], b_y [50] — all float CPU tensors.
    """
    ckpt = safe_torch_load(ckpt_path)
    sd   = ckpt.get('model_state_dict', ckpt)
    cfg  = ckpt.get('config', {})
    W_c  = sd['concept_encoder.weight'].float()   # [26, 2048]
    b_c  = sd['concept_encoder.bias'].float()     # [26]
    W_y  = sd['label_head.weight'].float()        # [50, 26]
    b_y  = sd['label_head.bias'].float()          # [50]
    return W_c, b_c, W_y, b_y, cfg


def load_mcbm_backbone(ckpt_path, device):
    """Extract only backbone from MCBM checkpoint, return eval-mode ResNet50."""
    ckpt = safe_torch_load(ckpt_path)
    sd   = ckpt.get('model_state_dict', ckpt)
    prefix = 'backbone.'
    backbone_state = {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}
    backbone = models.resnet50(weights=None)
    backbone.fc = nn.Identity()
    missing, unexpected = backbone.load_state_dict(backbone_state, strict=False)
    if missing:     print(f'  [backbone] missing: {missing}')
    if unexpected:  print(f'  [backbone] unexpected: {unexpected}')
    return backbone.to(device).eval()


print('Defined: load_mcbm_weights  load_mcbm_backbone')

### 5a. Checkpoint sanity check — verify stored γ matches filename

This answers: *"did the middle/small gammas actually get retrained?"*  
The checkpoint stores `config['gamma']`; if it doesn't match the filename gamma the model was not retrained.

In [ ]:
import datetime

print(f'{'gamma':>8s}  {'ckpt_exists':>12s}  {'stored_gamma':>13s}  {'match':>6s}  '
      f'{"sigma":>6s}  {"lambda_c":>9s}  mtime')
print('-' * 80)
for g in GAMMAS:
    p = mcbm_ckpt(g)
    if not p.exists():
        print(f'{g:>8.1f}  {"MISSING":>12s}')
        continue
    try:
        ckpt = safe_torch_load(p)
        cfg  = ckpt.get('config', {})
        stored_g  = cfg.get('gamma', 'N/A')
        stored_s  = cfg.get('sigma', 'N/A')
        stored_lc = cfg.get('lambda_c', 'N/A')
        match     = '✓' if abs(float(stored_g) - g) < 1e-6 else '✗ MISMATCH'
        mtime     = datetime.datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        print(f'{g:>8.1f}  {"OK":>12s}  {stored_g!s:>13s}  {match:>6s}  '
              f'{str(stored_s):>6s}  {str(stored_lc):>9s}  {mtime}')
    except Exception as e:
        print(f'{g:>8.1f}  ERROR loading: {e}')

print()
print('NOTE: If stored_gamma != filename gamma → model was NOT retrained with that gamma.')
print('Mismatched checkpoints should be retrained before drawing conclusions.')

## 6. MCBM inference functions

In [ ]:
def make_mcbm_inference_fns(W_c, b_c, W_y, b_y, backbone, device):
    """
    Returns (run_through_mcbm, z_to_probs_mcbm) closures bound to the given weights.
    
    z_raw = backbone(img) @ W_c.T + b_c   (no sigmoid — raw concept logits)
    y_logits = z_raw @ W_y.T + b_y
    probs = softmax(y_logits)
    """
    W_c_d = W_c.to(device)
    b_c_d = b_c.to(device)
    W_y_d = W_y.to(device)
    b_y_d = b_y.to(device)

    @torch.no_grad()
    def run_through_mcbm(img_pil: Image.Image):
        """PIL Image → (z_raw [26], probs [50]) both on CPU."""
        x   = eval_tf(img_pil).unsqueeze(0).to(device)
        avg = backbone(x)                                  # [1, 2048]
        z   = avg @ W_c_d.T + b_c_d                       # [1, 26] raw
        p   = torch.softmax(z @ W_y_d.T + b_y_d, dim=-1)  # [1, 50]
        return z.squeeze(0).cpu(), p.squeeze(0).cpu()

    @torch.no_grad()
    def z_to_probs_mcbm(z_raw: torch.Tensor) -> torch.Tensor:
        """z_raw [26] CPU → probs [50] CPU."""
        logits = z_raw @ W_y.T + b_y
        return torch.softmax(logits, dim=-1)

    return run_through_mcbm, z_to_probs_mcbm


@torch.no_grad()
def compute_z_from_avgpool(avg: torch.Tensor, W_c: torch.Tensor, b_c: torch.Tensor) -> torch.Tensor:
    """Batch: avg [N,2048] → z_raw [N,26] (CPU)."""
    return avg @ W_c.T + b_c


print('Defined: make_mcbm_inference_fns  compute_z_from_avgpool')

### 6a. Smoke test — verify pre-computed z matches live inference (one gamma)

In [ ]:
_g_test = 0.0   # change to any gamma to test a different one

if mcbm_ckpt(_g_test).exists() and mcbm_feats(_g_test).exists():
    _W_c, _b_c, _W_y, _b_y, _cfg = load_mcbm_weights(mcbm_ckpt(_g_test))
    print(f'concept_encoder: {tuple(_W_c.shape)}  label_head: {tuple(_W_y.shape)}')
    print(f'config: {_cfg}')

    _ids_te  = load_split_order(mcbm_feats(_g_test), 'test')
    _avg_te  = load_features(mcbm_feats(_g_test), 'avgpool', 'test')
    _z_te    = compute_z_from_avgpool(_avg_te, _W_c, _b_c)   # raw z

    _id_to_row = {int(i): r for r, i in enumerate(_ids_te)}
    _sids_te   = np.array([int(_meta_idx.loc[int(i), 'species_id']) for i in _ids_te])

    # Task accuracy from raw z
    _logits_te = _z_te @ _W_y.T + _b_y
    _pred_sp   = _logits_te.argmax(dim=1).numpy()
    _sp_acc    = float((_pred_sp == _sids_te).mean())
    print(f'gamma={_g_test}: species accuracy from pre-computed z = {_sp_acc:.4f}')

    # Concept accuracy (using sigmoid threshold on raw z)
    _gt_c   = cc_matrix[_sids_te]
    _pred_c = (torch.sigmoid(_z_te).numpy() > 0.5).astype(int)
    print(f'Concept accuracy (sigmoid threshold 0.5): {(_pred_c == _gt_c).mean():.4f}')
    print(f'avgpool_te: {tuple(_avg_te.shape)}  z_te: {tuple(_z_te.shape)}')

    del _W_c, _b_c, _W_y, _b_y, _avg_te, _z_te
else:
    print(f'[skip] gamma={_g_test} checkpoint or features not found.')

## 7. Part swap helpers  *(identical to v2)*

In [ ]:
def swap_part_in_ann(ann: dict, part: str, new_params: dict) -> dict:
    cf = dict(ann)
    cf[f'{part}_model'] = new_params['model']
    if part in PARTS_WITH_COLOR:
        cf[f'{part}_color'] = new_params.get('color', '')
    return cf


def delete_part_in_ann(ann: dict, part: str) -> dict:
    cf = dict(ann)
    cf[f'{part}_model'] = ''
    if part in PARTS_WITH_COLOR:
        cf[f'{part}_color'] = ''
    return cf


def get_concept_dims(part: str, var_old: int, var_new: int):
    c_old = CONCEPT_TO_IDX.get(f'{part}_{var_old}', -1)
    c_new = CONCEPT_TO_IDX.get(f'{part}_{var_new}', -1)
    return c_old, c_new


print('Defined: swap_part_in_ann  delete_part_in_ann  get_concept_dims')

## 8. Z-ordering record (MCBM-adapted)

Same logic as v2 but:
- z_raw has no sigmoid applied
- GT ceiling: z_raw[c_donor]=+3, z_raw[c_src]=-3 (IB q_phi saturation range)
- `z_new_orig` and `z_old_orig` are raw values (not sigmoid-bounded)

In [ ]:
def z_ordering_record_mcbm(
    ann_orig: dict,
    sid_src: int,
    part: str,
    var_src: int,
    sid_donor: int,
    var_donor: int,
    z_orig: torch.Tensor,    # pre-computed z_raw for this image [26]
    run_through_mcbm,        # closure: PIL -> (z_raw, probs)
    z_to_probs_mcbm,         # closure: z_raw -> probs
) -> dict:
    """
    Render source image with donor part, run through MCBM.
    z_raw ordering: z_cf[c_donor] > z_cf[c_src]  (correct = True)
    GT ceiling: set z[c_donor]=MCBM_Z_ACTIVE, z[c_src]=MCBM_Z_INACTIVE.
    """
    c_src   = CONCEPT_TO_IDX[f'{part}_{var_src}']
    c_donor = CONCEPT_TO_IDX[f'{part}_{var_donor}']

    ann_cf       = swap_part_in_ann(ann_orig, part, species_part_params[sid_donor][part])
    img_cf       = render_ann(ann_cf)
    z_cf, p_cf   = run_through_mcbm(img_cf)

    z_new  = float(z_cf[c_donor])
    z_old  = float(z_cf[c_src])
    margin = z_new - z_old

    z_gt          = z_orig.clone()
    z_gt[c_donor] = MCBM_Z_ACTIVE
    z_gt[c_src]   = MCBM_Z_INACTIVE
    p_gt          = z_to_probs_mcbm(z_gt)

    row = {
        'sid_src':          sid_src,
        'sid_donor':        sid_donor,
        'part':             part,
        'var_src':          var_src,
        'var_donor':        var_donor,
        'c_src':            c_src,
        'c_donor':          c_donor,
        'z_new':            z_new,
        'z_old':            z_old,
        'z_new_orig':       float(z_orig[c_donor]),
        'z_old_orig':       float(z_orig[c_src]),
        'margin':           margin,
        'ordering_correct': bool(margin > 0),
        'p_cf_donor':       float(p_cf[sid_donor]),
        'p_gt_donor':       float(p_gt[sid_donor]),
    }
    if part == 'tail':
        for i in range(PART_VARIANTS['tail']):
            row[f'z_cf_tail_{i}'] = float(z_cf[CONCEPT_TO_IDX[f'tail_{i}']])
    return row


print('Defined: z_ordering_record_mcbm')

In [ ]:
import threading, itertools as _itools

PART_SEG_COLORS = {
    'beak': (255, 255,   0),
    'eye':  (255, 255, 253),
    'wing': (  0, 255,   1),
    'foot': (255,   0,   1),
    'tail': (  0,   0, 255),
}
_port_lock  = threading.Lock()
_port_cycle = _itools.cycle([8081])

def render_part_map(ann: dict) -> Image.Image:
    with _port_lock:
        port = next(_port_cycle)
    url = json_to_url(ann, prefix=f'http://localhost:{port}/render?', render_mode='part_map')
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    img_bytes = decodebytes(resp.content)
    return Image.open(io.BytesIO(img_bytes)).convert('RGB').resize((256, 256), Image.NEAREST)


def part_pixel_count(img_seg: Image.Image, part: str) -> int:
    arr = np.array(img_seg)
    r, g, b = PART_SEG_COLORS[part]
    mask = (arr[:, :, 0] == r) & (arr[:, :, 1] == g) & (arr[:, :, 2] == b)
    return int(mask.sum())


def z_ordering_record_mcbm_v2(
    ann_orig, sid_src, part, var_src, sid_donor, var_donor, z_orig,
    run_through_mcbm, z_to_probs_mcbm
) -> dict:
    """Like z_ordering_record_mcbm but adds pixel_count_cf via part_map render."""
    c_src   = CONCEPT_TO_IDX[f'{part}_{var_src}']
    c_donor = CONCEPT_TO_IDX[f'{part}_{var_donor}']

    ann_cf     = swap_part_in_ann(ann_orig, part, species_part_params[sid_donor][part])
    img_cf     = render_ann(ann_cf)
    img_seg    = render_part_map(ann_cf)
    z_cf, p_cf = run_through_mcbm(img_cf)

    z_new  = float(z_cf[c_donor])
    z_old  = float(z_cf[c_src])
    margin = z_new - z_old

    z_gt          = z_orig.clone()
    z_gt[c_donor] = MCBM_Z_ACTIVE
    z_gt[c_src]   = MCBM_Z_INACTIVE
    p_gt          = z_to_probs_mcbm(z_gt)

    row = {
        'sid_src': sid_src, 'sid_donor': sid_donor, 'part': part,
        'var_src': var_src, 'var_donor': var_donor,
        'c_src': c_src, 'c_donor': c_donor,
        'z_new': z_new, 'z_old': z_old,
        'z_new_orig': float(z_orig[c_donor]),
        'z_old_orig': float(z_orig[c_src]),
        'margin': margin,
        'ordering_correct': bool(margin > 0),
        'p_cf_donor': float(p_cf[sid_donor]),
        'p_gt_donor': float(p_gt[sid_donor]),
        'pixel_count_cf': part_pixel_count(img_seg, part),
    }
    if part == 'tail':
        for i in range(PART_VARIANTS['tail']):
            row[f'z_cf_tail_{i}'] = float(z_cf[CONCEPT_TO_IDX[f'tail_{i}']])
    return row

print('Defined: render_part_map  part_pixel_count  z_ordering_record_mcbm_v2')

## 9. Define species pairs  *(once; reused across all gammas)*

In [ ]:
rng = random.Random(42)

all_pairs = {}   # part -> list of (sid_A, var_A, sid_B, var_B)
for part in PARTS:
    pairs = [
        (sid_A, species_variant_idx[sid_A][part],
         sid_B, species_variant_idx[sid_B][part])
        for sid_A, sid_B in combinations(range(N_SPECIES), 2)
        if species_variant_idx[sid_A][part] != species_variant_idx[sid_B][part]
    ]
    if MAX_PAIRS_PER_PART is not None and len(pairs) > MAX_PAIRS_PER_PART:
        pairs = rng.sample(pairs, MAX_PAIRS_PER_PART)
    all_pairs[part] = pairs
    print(f'{part}: {len(pairs)} pairs  (~{len(pairs) * 2 * MAX_IMGS_PER_SPECIES} renders per gamma)')

total_renders_per_gamma = sum(len(p) * 2 * MAX_IMGS_PER_SPECIES for p in all_pairs.values())
print(f'\nTotal renders per gamma: {total_renders_per_gamma}  (~{total_renders_per_gamma*1.5/60:.0f} min at 1.5s each)')
print(f'Total renders for all {len(GAMMAS)} gammas: {total_renders_per_gamma * len(GAMMAS)}')

## 10. Renderer visual check  *(optional — run once to verify renderer)*

In [ ]:
if not RENDERER_AVAILABLE:
    print('[skip] Renderer not available.')
else:
    _vsid  = 0
    _v_idx = test_idx_by_species[_vsid][0]
    _v_ann = test_anns[_v_idx]

    _donors = {}
    for _part in PARTS:
        _var_A = species_variant_idx[_vsid][_part]
        for _sid_B in range(N_SPECIES):
            if _sid_B != _vsid and species_variant_idx[_sid_B][_part] != _var_A:
                _donors[_part] = _sid_B
                break

    _nrows = len(PARTS)
    fig, axes = plt.subplots(_nrows, 3, figsize=(10, _nrows * 2.8))
    _col_titles = ['Original', 'Swap (part replaced)', 'Deletion (part removed)']
    for ci, ct in enumerate(_col_titles):
        axes[0, ci].set_title(ct, fontsize=10, fontweight='bold', pad=6)

    for ri, _part in enumerate(PARTS):
        _color  = PART_COLORS[_part]
        _donor  = _donors.get(_part)
        _orig   = render_ann(_v_ann)
        _ann_sw = swap_part_in_ann(_v_ann, _part, species_part_params[_donor][_part]) if _donor else None
        _img_sw = render_ann(_ann_sw) if _ann_sw else None
        _ann_dl = delete_part_in_ann(_v_ann, _part)
        _img_dl = render_ann(_ann_dl)

        axes[ri, 0].set_ylabel(_part, fontsize=10, rotation=0, ha='right', labelpad=40,
                               color=_color, fontweight='bold')
        axes[ri, 0].imshow(_orig)
        if _img_sw is not None:
            axes[ri, 1].imshow(_img_sw)
        axes[ri, 2].imshow(_img_dl)
        for ci in range(3):
            axes[ri, ci].axis('off')
            for spine in axes[ri, ci].spines.values():
                spine.set_edgecolor(_color); spine.set_linewidth(1.5)

    plt.suptitle(f'Renderer visual check — species {_vsid} ({spname(_vsid)})', y=1.01, fontsize=10)
    plt.tight_layout()
    plt.savefig('fb_mcbm_renderer_visual_grid.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_mcbm_renderer_visual_grid.png')

## 11. Main gamma sweep — Z-ordering experiment

For each γ, run the z-ordering experiment (tail first, then all parts).  
Results cached to CSV: `fb_mcbm_z_ordering_gamma{g}.csv` — re-run only if missing.

**To force re-run:** delete the CSV or set `FORCE_RERUN = True`.

In [ ]:
FORCE_RERUN = False   # set True to re-run even if CSV exists
USE_V2      = True    # include pixel_count_cf (part_map renders, 2x slower but needed for occlusion analysis)


def run_z_ordering_for_gamma(g, use_v2=False):
    """
    Run z-ordering experiment for gamma=g.
    Returns DataFrame with one row per (image, part, pair direction).
    """
    ckpt_path  = mcbm_ckpt(g)
    feats_path = mcbm_feats(g)

    if not ckpt_path.exists():
        print(f'  [skip] gamma={g}: checkpoint not found')
        return None
    if not feats_path.exists():
        print(f'  [skip] gamma={g}: features not found')
        return None

    print(f'\n=== gamma={g} ===')
    W_c, b_c, W_y, b_y, cfg = load_mcbm_weights(ckpt_path)
    backbone = load_mcbm_backbone(ckpt_path, device)
    run_fn, z2p_fn = make_mcbm_inference_fns(W_c, b_c, W_y, b_y, backbone, device)

    ids_te  = load_split_order(feats_path, 'test')
    avg_te  = load_features(feats_path, 'avgpool', 'test')
    z_te    = compute_z_from_avgpool(avg_te, W_c, b_c)   # [N, 26] raw
    id_to_row = {int(i): r for r, i in enumerate(ids_te)}

    record_fn = z_ordering_record_mcbm_v2 if use_v2 else z_ordering_record_mcbm
    rows = []

    for part in PARTS:
        pairs = all_pairs[part]
        for sid_A, var_A, sid_B, var_B in tqdm(pairs, desc=f'  {part}'):
            for local_idx in test_idx_by_species[sid_A][:MAX_IMGS_PER_SPECIES]:
                gid = _FUNNYBIRDS_N_TRAIN + local_idx
                row_idx = id_to_row.get(gid)
                if row_idx is None: continue
                r = record_fn(test_anns[local_idx], sid_A, part, var_A,
                              sid_B, var_B, z_te[row_idx], run_fn, z2p_fn)
                r['direction'] = 'fwd'
                rows.append(r)

            for local_idx in test_idx_by_species[sid_B][:MAX_IMGS_PER_SPECIES]:
                gid = _FUNNYBIRDS_N_TRAIN + local_idx
                row_idx = id_to_row.get(gid)
                if row_idx is None: continue
                r = record_fn(test_anns[local_idx], sid_B, part, var_B,
                              sid_A, var_A, z_te[row_idx], run_fn, z2p_fn)
                r['direction'] = 'bwd'
                rows.append(r)

        _part_df = pd.DataFrame([r for r in rows if r['part'] == part])
        if len(_part_df):
            print(f'    {part}: {len(_part_df)} rows  '
                  f'ordering_correct={_part_df["ordering_correct"].mean():.3%}  '
                  f'mean_margin={_part_df["margin"].mean():+.4f}')

    # Clean up GPU memory
    del backbone, W_c, b_c, W_y, b_y
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.DataFrame(rows)


print('Defined: run_z_ordering_for_gamma')

In [ ]:
if not RENDERER_AVAILABLE:
    print('[SKIP] Renderer not available — start Node.js server and re-run.')
else:
    for g in GAMMAS:
        suffix   = '_v2' if USE_V2 else ''
        csv_path = Path(f'fb_mcbm_z_ordering_gamma{g}{suffix}.csv')
        if csv_path.exists() and not FORCE_RERUN:
            print(f'[cache] gamma={g}: {csv_path} already exists, skipping.')
            continue
        df_g = run_z_ordering_for_gamma(g, use_v2=USE_V2)
        if df_g is not None:
            df_g.to_csv(csv_path, index=False)
            print(f'  Saved {csv_path}  ({len(df_g)} rows)')

## 12. Load all gamma results

In [ ]:
suffix = '_v2' if USE_V2 else ''
gamma_dfs = {}   # g -> DataFrame
for g in GAMMAS:
    p = Path(f'fb_mcbm_z_ordering_gamma{g}{suffix}.csv')
    if p.exists():
        gamma_dfs[g] = pd.read_csv(p)
        print(f'Loaded gamma={g}: {len(gamma_dfs[g])} rows')
    else:
        print(f'[missing] gamma={g}: {p}')

print(f'\nGammas available: {sorted(gamma_dfs.keys())}')

## 13. Per-gamma summary tables

In [ ]:
def make_part_summary(df):
    return (
        df.groupby('part', as_index=False)
        .agg(
            n_images        = ('ordering_correct', 'size'),
            frac_correct    = ('ordering_correct', 'mean'),
            frac_violations = ('ordering_correct', lambda x: 1 - x.mean()),
            mean_margin     = ('margin', 'mean'),
            std_margin      = ('margin', 'std'),
            mean_p_cf_donor = ('p_cf_donor', 'mean'),
            mean_p_gt_donor = ('p_gt_donor', 'mean'),
        )
        .sort_values('frac_violations', ascending=False)
        .reset_index(drop=True)
    )


gamma_summaries = {}   # g -> per-part summary DataFrame

for g, df in gamma_dfs.items():
    smry = make_part_summary(df)
    gamma_summaries[g] = smry
    print(f'\n=== gamma={g} ===')
    display(smry[['part','n_images','frac_correct','frac_violations','mean_margin']])

In [ ]:
# Per-gamma concept breakdown
gamma_concept_summaries = {}

for g, df in gamma_dfs.items():
    cs = (
        df.groupby(['part', 'c_donor'], as_index=False)
        .agg(
            n_images     = ('ordering_correct', 'size'),
            frac_correct = ('ordering_correct', 'mean'),
            mean_margin  = ('margin', 'mean'),
        )
    )
    cs['concept'] = cs['c_donor'].apply(lambda i: CONCEPT_NAMES[int(i)])
    cs = cs.sort_values('frac_correct').reset_index(drop=True)
    gamma_concept_summaries[g] = cs

if gamma_concept_summaries:
    g0 = sorted(gamma_concept_summaries.keys())[0]
    print(f'Concept summary for gamma={g0} (worst first):')
    display(gamma_concept_summaries[g0].head(15))

## 14. Cross-gamma comparison plots

In [ ]:
# Table: frac_correct per part per gamma
if gamma_summaries:
    rows = []
    for g, smry in sorted(gamma_summaries.items()):
        for _, r in smry.iterrows():
            rows.append({'gamma': g, 'part': r['part'],
                         'frac_correct': r['frac_correct'],
                         'frac_violations': r['frac_violations'],
                         'mean_margin': r['mean_margin'],
                         'n_images': r['n_images']})
    cross_gamma_df = pd.DataFrame(rows)

    pivot_fc = cross_gamma_df.pivot(index='gamma', columns='part', values='frac_correct').round(4)
    pivot_mv = cross_gamma_df.pivot(index='gamma', columns='part', values='mean_margin').round(4)

    print('frac_correct per part per gamma (higher = better grounding):')
    display(pivot_fc)
    print('\nmean_margin per part per gamma (higher = more confident correct ordering):')
    display(pivot_mv)

In [ ]:
# Line plots: frac_correct vs gamma per part
if gamma_summaries:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for part in PARTS:
        sub = cross_gamma_df[cross_gamma_df['part'] == part].sort_values('gamma')
        if sub.empty: continue
        axes[0].plot(sub['gamma'], sub['frac_correct'],
                     marker='o', color=PART_COLORS[part], label=part)
        axes[1].plot(sub['gamma'], sub['mean_margin'],
                     marker='o', color=PART_COLORS[part], label=part)

    axes[0].axhline(0.5, color='gray', ls='--', lw=1.5, label='chance')
    axes[0].set_xlabel('gamma (IB penalty weight)')
    axes[0].set_ylabel('frac_correct')
    axes[0].set_title('Z-ordering accuracy vs gamma\n(higher = better visual grounding)')
    axes[0].legend(fontsize=8)
    axes[0].set_ylim(0, 1)
    axes[0].grid(True, alpha=0.3)

    axes[1].axhline(0, color='gray', ls='--', lw=1.5)
    axes[1].set_xlabel('gamma (IB penalty weight)')
    axes[1].set_ylabel('mean margin  (z_new − z_old)')
    axes[1].set_title('Mean z-ordering margin vs gamma\n(higher = concept bottleneck more decisive)')
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('MCBM: Z-ordering experiment — cross-gamma comparison', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig('fb_mcbm_frac_correct_vs_gamma.pdf', bbox_inches='tight')
    plt.savefig('fb_mcbm_frac_correct_vs_gamma.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_mcbm_frac_correct_vs_gamma.pdf/png')

In [ ]:
# Heatmap: frac_violations per part per gamma
if gamma_summaries:
    import matplotlib.colors as mcolors
    pivot_fv = cross_gamma_df.pivot(index='gamma', columns='part', values='frac_violations')

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(pivot_fv.values, cmap='RdYlGn_r', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(pivot_fv.columns)))
    ax.set_xticklabels(pivot_fv.columns)
    ax.set_yticks(range(len(pivot_fv.index)))
    ax.set_yticklabels([f'γ={g}' for g in pivot_fv.index])
    ax.set_xlabel('Part')
    ax.set_ylabel('Gamma')
    ax.set_title('Violation rate (frac_violations) per part per gamma\n'
                 'Red = high backwash, Green = well-grounded')
    for i in range(len(pivot_fv.index)):
        for j in range(len(pivot_fv.columns)):
            v = pivot_fv.values[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if v > 0.6 else 'black')
    plt.colorbar(im, ax=ax, label='violation rate')
    plt.tight_layout()
    plt.savefig('fb_mcbm_violation_heatmap.pdf', bbox_inches='tight')
    plt.savefig('fb_mcbm_violation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_mcbm_violation_heatmap.pdf/png')

In [ ]:
# Top-20 worst concepts per gamma (bar chart grid)
if gamma_concept_summaries:
    n_gammas = len(gamma_concept_summaries)
    fig, axes = plt.subplots(1, n_gammas, figsize=(6 * n_gammas, 6), sharey=False)
    if n_gammas == 1: axes = [axes]

    for ax, (g, cs) in zip(axes, sorted(gamma_concept_summaries.items())):
        top = cs.sort_values('frac_correct').head(20)
        colors = [PART_COLORS[CONCEPT_TO_PART[c]] for c in top['concept']]
        ax.barh(top['concept'], 1 - top['frac_correct'], color=colors, alpha=0.8)
        ax.axvline(0.5, color='gray', ls='--', alpha=0.5, label='chance')
        ax.set_xlabel('Violation rate')
        ax.set_title(f'γ={g}\nTop-20 concepts by violation rate', fontsize=9)
        ax.set_xlim(0, 1)
        ax.grid(True, axis='x', alpha=0.3)

    legend_patches = [Patch(color=c, label=p) for p, c in PART_COLORS.items()]
    axes[-1].legend(handles=legend_patches, fontsize=7, loc='lower right')
    plt.suptitle('MCBM: Worst-grounded concept dims by gamma', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig('fb_mcbm_violation_by_concept_gamma.pdf', bbox_inches='tight')
    plt.savefig('fb_mcbm_violation_by_concept_gamma.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_mcbm_violation_by_concept_gamma.pdf/png')

## 15. Detailed per-gamma analysis

Same detailed plots as `fb_cbm_renderer_swap_v2.ipynb` but parameterised over gamma.  
Set `DETAIL_GAMMA` to the gamma you want to inspect.

In [ ]:
DETAIL_GAMMA = 0.1   # change to any available gamma

if DETAIL_GAMMA not in gamma_dfs:
    print(f'[skip] gamma={DETAIL_GAMMA} not loaded. Available: {sorted(gamma_dfs.keys())}')
else:
    z_order_df = gamma_dfs[DETAIL_GAMMA].copy()
    part_summary = gamma_summaries[DETAIL_GAMMA].copy()
    print(f'Using gamma={DETAIL_GAMMA}  ({len(z_order_df)} rows)')

    overall = z_order_df['ordering_correct'].mean()
    n_total = len(z_order_df)
    n_viol  = int((~z_order_df['ordering_correct']).sum())
    worst   = part_summary.iloc[0]
    best    = part_summary.iloc[-1]
    print('=' * 55)
    print(f'  HEADLINE: z-ordering correct {overall:.1%} ({n_total - n_viol}/{n_total})')
    print(f'  Violations: {n_viol} / {n_total} ({1-overall:.1%})')
    print(f'  Most violated: {worst["part"]} ({worst["frac_violations"]:.1%})')
    print(f'  Best grounded: {best["part"]}  ({best["frac_violations"]:.1%})')
    print('=' * 55)

In [ ]:
# Per-part z_old_orig and z_new_orig boxplots
# NOTE: For MCBM, values are raw (not sigmoid-bounded); use sigmoid(z) for probability scale
if DETAIL_GAMMA in gamma_dfs:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for ax, col, title in [
        (axes[0], 'z_old_orig',
         f'z_old_orig per part (raw)  γ={DETAIL_GAMMA}\n(source concept in original; should be high/positive)'),
        (axes[1], 'z_new_orig',
         f'z_new_orig per part (raw)  γ={DETAIL_GAMMA}\n(donor concept in original; should be low/negative)'),
    ]:
        data = [z_order_df[z_order_df['part'] == p][col].values for p in PARTS]
        bp   = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', lw=2))
        for patch, part in zip(bp['boxes'], PARTS):
            patch.set_facecolor(PART_COLORS[part]); patch.set_alpha(0.7)
        ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.7)
        ax.set_xticks(range(1, len(PARTS)+1))
        ax.set_xticklabels(PARTS)
        ax.set_ylabel(col + '  (raw z_raw)')
        ax.set_title(title, fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)

    plt.suptitle(f'Concept activations in ORIGINAL images  γ={DETAIL_GAMMA}  (MCBM raw z, no sigmoid)', y=1.02)
    plt.tight_layout()
    plt.savefig(f'fb_mcbm_z_grounding_boxplots_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Tail margin distribution per concept dim
if DETAIL_GAMMA in gamma_dfs:
    tail_df = z_order_df[z_order_df['part'] == 'tail'].copy()
    n_tail  = PART_VARIANTS['tail']
    fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=False)
    axes = axes.flatten()
    for i in range(n_tail):
        ax   = axes[i]
        cidx = CONCEPT_TO_IDX[f'tail_{i}']
        sub  = tail_df[tail_df['c_donor'] == cidx]
        if sub.empty:
            ax.set_visible(False); continue
        fc = sub['ordering_correct'].mean()
        ax.hist(sub['margin'], bins=30, color=PART_COLORS['tail'], alpha=0.75, edgecolor='white')
        ax.axvline(0, color='red', ls='--', lw=1.5)
        ax.set_title(f'tail_{i}  frac_correct={fc:.2%}', fontsize=9)
        ax.set_xlabel('margin (z_new − z_old, raw)', fontsize=8)
        ax.set_ylabel('count', fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.suptitle(f'Tail margin per concept dim  γ={DETAIL_GAMMA}  (red=violation threshold)', y=1.01, fontsize=11)
    plt.tight_layout()
    plt.savefig(f'fb_mcbm_z_tail_concept_margins_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Tail concept confusion matrix (violations only)
if DETAIL_GAMMA in gamma_dfs:
    n_tail    = PART_VARIANTS['tail']
    tail_df   = z_order_df[z_order_df['part'] == 'tail'].copy()
    tail_viol = tail_df[~tail_df['ordering_correct']].copy()

    if tail_viol.empty:
        print(f'No tail violations for gamma={DETAIL_GAMMA} — perfect grounding!')
    else:
        z_cf_tail_cols = [f'z_cf_tail_{i}' for i in range(n_tail)]
        missing = [c for c in z_cf_tail_cols if c not in tail_viol.columns]
        if missing:
            print(f'[WARNING] Missing columns {missing}. Re-run with USE_V2=True or check record function.')
        else:
            argmax_dim = tail_viol[z_cf_tail_cols].idxmax(axis=1).str.extract(r'(\d+)')[0].astype(int)
            tail_viol['argmax_tail_dim'] = argmax_dim.values
            tail_viol['is_anchoring']    = tail_viol['argmax_tail_dim'] == tail_viol['var_src']

            n_anch  = tail_viol['is_anchoring'].sum()
            n_total_v = len(tail_viol)
            print(f'Violations: {n_total_v}')
            print(f'Anchoring (model stayed on original tail): {n_anch} ({n_anch/n_total_v:.1%})')
            print(f'Genuine confusion:                        {n_total_v-n_anch} ({(n_total_v-n_anch)/n_total_v:.1%})')

            def build_conf(df):
                c = np.zeros((n_tail, n_tail), dtype=int)
                for _, r in df.iterrows():
                    c[int(r['var_donor']), int(r['argmax_tail_dim'])] += 1
                return c

            conf_raw  = build_conf(tail_viol)
            conf_filt = build_conf(tail_viol[~tail_viol['is_anchoring']])

            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            for ax, conf, title, sub in [
                (axes[0], conf_raw,  'Raw violations',
                 f'All {n_total_v} violations (diagonal = anchoring)'),
                (axes[1], conf_filt, 'Filtered — genuine confusion',
                 f'Anchoring removed  (n={n_total_v - n_anch})'),
            ]:
                im = ax.imshow(conf, cmap='Reds')
                ax.set_xticks(range(n_tail)); ax.set_yticks(range(n_tail))
                ax.set_xticklabels([f'tail_{i}' for i in range(n_tail)], rotation=45, ha='right', fontsize=8)
                ax.set_yticklabels([f'tail_{i}' for i in range(n_tail)], fontsize=8)
                ax.set_xlabel('argmax z_cf tail dim'); ax.set_ylabel('var_donor')
                ax.set_title(f'{title}\n{sub}', fontsize=9)
                for ii in range(n_tail):
                    for jj in range(n_tail):
                        v = conf[ii, jj]
                        if v > 0:
                            ax.text(jj, ii, str(v), ha='center', va='center', fontsize=8,
                                    color='white' if v > conf.max()*.5 else 'black')
                plt.colorbar(im, ax=ax, label='count')

            plt.suptitle(f'Concept confusion — tail violations  γ={DETAIL_GAMMA}', y=1.02, fontsize=10)
            plt.tight_layout()
            plt.savefig(f'fb_mcbm_z_tail_confusion_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
            plt.show()

In [ ]:
# Source species violation rate (tail) — per-gamma
if DETAIL_GAMMA in gamma_dfs:
    tail_df = z_order_df[z_order_df['part'] == 'tail'].copy()
    viol_by_src = (
        tail_df.groupby('sid_src')
        .agg(n_images=('ordering_correct','size'),
             n_violations=('ordering_correct', lambda x: (~x).sum()),
             frac_violations=('ordering_correct', lambda x: 1 - x.mean()))
        .reset_index().sort_values('frac_violations', ascending=False)
    )
    med = viol_by_src['frac_violations'].median()
    colors = ['crimson' if v > med else 'steelblue' for v in viol_by_src['frac_violations']]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(viol_by_src['sid_src'].astype(str), viol_by_src['frac_violations'],
           color=colors, alpha=0.8)
    ax.axhline(med, color='black', ls='--', lw=1.5, label=f'median ({med:.2%})')
    ax.set_xlabel('sid_src'); ax.set_ylabel('frac_violations')
    ax.set_title(f'Tail violation rate by source species  γ={DETAIL_GAMMA}  (red = above median)')
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
    plt.xticks(rotation=90, fontsize=7)
    plt.tight_layout()
    plt.savefig(f'fb_mcbm_z_viol_by_sid_src_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(viol_by_src.head(10))

In [ ]:
# GT ceiling vs render-swap donor probability (tail)
if DETAIL_GAMMA in gamma_dfs:
    tail_df = z_order_df[z_order_df['part'] == 'tail']
    colors  = tail_df['ordering_correct'].map({True: 'steelblue', False: 'crimson'})

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    # margin vs p_cf_donor
    axes[0].scatter(tail_df['margin'], tail_df['p_cf_donor'], c=colors, s=14, alpha=0.4)
    axes[0].axvline(0, color='black', ls='--', lw=1)
    axes[0].set_xlabel('margin  (z_new − z_old)')
    axes[0].set_ylabel('p_cf_donor')
    axes[0].set_title(f'Tail: margin vs donor probability  γ={DETAIL_GAMMA}')
    axes[0].grid(True, alpha=0.3)

    # p_gt_donor vs p_cf_donor
    _lim = max(tail_df['p_gt_donor'].max(), tail_df['p_cf_donor'].max()) * 1.05
    axes[1].scatter(tail_df['p_gt_donor'], tail_df['p_cf_donor'], c=colors, s=14, alpha=0.4)
    axes[1].plot([0, _lim], [0, _lim], 'k--', alpha=0.4, lw=1, label='perfect grounding')
    axes[1].set_xlabel('p_gt_donor  (GT ceiling: z=+3/-3 intervention)')
    axes[1].set_ylabel('p_cf_donor  (render swap)')
    axes[1].set_title(f'GT ceiling vs render swap  γ={DETAIL_GAMMA}\nBelow diagonal = missed visual evidence')
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    for ax in axes:
        ax.legend(handles=[
            Line2D([0],[0],marker='o',color='w',markerfacecolor='steelblue',ms=8,label='correct'),
            Line2D([0],[0],marker='o',color='w',markerfacecolor='crimson', ms=8,label='violation'),
        ] + ([Line2D([0],[0],linestyle='--',color='black',label='perfect grounding')] if ax == axes[1] else []),
                  fontsize=8)

    plt.tight_layout()
    plt.savefig(f'fb_mcbm_z_margin_p_cf_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 16. Part 2 — Segmentation / Occlusion Analysis

Tests whether violations are occlusion-driven or backwash-driven.  
Requires `USE_V2=True` (pixel_count_cf column present).  
Runs for `DETAIL_GAMMA`; can compare two gammas at end.

In [ ]:
if DETAIL_GAMMA not in gamma_dfs or 'pixel_count_cf' not in gamma_dfs[DETAIL_GAMMA].columns:
    print(f'[skip] pixel_count_cf not available. Re-run with USE_V2=True.')
else:
    z_order_df_v2 = gamma_dfs[DETAIL_GAMMA].copy()

    # Pixel count histogram per part
    fig, axes = plt.subplots(1, len(PARTS), figsize=(4 * len(PARTS), 3.5))
    for ax, part in zip(axes, PARTS):
        sub = z_order_df_v2[z_order_df_v2['part'] == part]['pixel_count_cf']
        ax.hist(sub, bins=40, color=PART_COLORS[part], alpha=0.8, edgecolor='white')
        ax.axvline(sub.median(), color='black', ls='--', lw=1.5, label=f'median={sub.median():.0f}')
        ax.set_title(f'{part}  γ={DETAIL_GAMMA}', fontsize=10)
        ax.set_xlabel('pixel_count_cf', fontsize=8)
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)
    plt.suptitle('Visibility of swapped-in part in counterfactual render', y=1.02)
    plt.tight_layout()
    plt.savefig(f'fb_mcbm_seg_pixel_hist_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if DETAIL_GAMMA in gamma_dfs and 'pixel_count_cf' in gamma_dfs[DETAIL_GAMMA].columns:
    tail_v2 = z_order_df_v2[z_order_df_v2['part'] == 'tail'].copy()
    colors  = tail_v2['ordering_correct'].map({True: 'steelblue', False: 'crimson'})

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Scatter: pixel_count_cf vs margin
    axes[0].scatter(tail_v2['pixel_count_cf'], tail_v2['margin'], c=colors, s=14, alpha=0.4)
    axes[0].axhline(0, color='black', ls='--', lw=1)
    axes[0].set_xlabel('pixel_count_cf'); axes[0].set_ylabel('margin')
    axes[0].set_title(f'Tail: visibility vs margin  γ={DETAIL_GAMMA}')
    axes[0].grid(True, alpha=0.3)

    # Violin: pixel_count_cf by correct/violation
    tail_v2['outcome'] = tail_v2['ordering_correct'].map({True: 'correct', False: 'violation'})
    for i, (label, color) in enumerate([('correct','steelblue'),('violation','crimson')]):
        sub = tail_v2[tail_v2['outcome'] == label]['pixel_count_cf']
        parts_vp = axes[1].violinplot(sub, positions=[i], showmedians=True)
        for pc in parts_vp['bodies']:
            pc.set_facecolor(color); pc.set_alpha(0.7)
    axes[1].set_xticks([0,1]); axes[1].set_xticklabels(['correct','violation'])
    axes[1].set_ylabel('pixel_count_cf')
    axes[1].set_title(f'Tail: visibility by outcome  γ={DETAIL_GAMMA}\n'
                      'Overlapping → violations not purely occlusion-driven')
    axes[1].grid(True, axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'fb_mcbm_seg_violin_gamma{DETAIL_GAMMA}.png', dpi=150, bbox_inches='tight')
    plt.show()

    from scipy.stats import mannwhitneyu
    grp_c = tail_v2[tail_v2['outcome']=='correct']['pixel_count_cf']
    grp_v = tail_v2[tail_v2['outcome']=='violation']['pixel_count_cf']
    u_stat, p_val = mannwhitneyu(grp_c, grp_v, alternative='greater')
    print(f'Mann-Whitney U (correct > violation in pixel_count): U={u_stat:.0f}  p={p_val:.4f}')
    print(f'  correct  median px: {grp_c.median():.0f}')
    print(f'  violation median px: {grp_v.median():.0f}')

In [ ]:
# Before vs after occlusion filter — for each gamma with pixel_count_cf
MIN_PART_PIXELS = 50

filter_comparison_rows = []

for g, df in sorted(gamma_dfs.items()):
    if 'pixel_count_cf' not in df.columns:
        continue
    for part in PARTS:
        sub     = df[df['part'] == part]
        sub_flt = sub[sub['pixel_count_cf'] >= MIN_PART_PIXELS]
        filter_comparison_rows.append({
            'gamma': g, 'part': part,
            'frac_correct_all':      sub['ordering_correct'].mean() if len(sub) else float('nan'),
            'frac_correct_filtered': sub_flt['ordering_correct'].mean() if len(sub_flt) else float('nan'),
            'n_all':      len(sub),
            'n_filtered': len(sub_flt),
        })

if filter_comparison_rows:
    fc_df = pd.DataFrame(filter_comparison_rows)
    fc_df['delta'] = fc_df['frac_correct_filtered'] - fc_df['frac_correct_all']

    print(f'Occlusion filter (px >= {MIN_PART_PIXELS}): frac_correct before vs after')
    display(fc_df.pivot_table(index='gamma', columns='part', values='delta').round(4))
    print('\nPositive delta = occlusion filter improves grounding (some violations were occlusion-driven)')
    print('Near-zero delta = violations are backwash, not occlusion')
else:
    print('[skip] No gammas with pixel_count_cf available.')

In [ ]:
# Plot: frac_correct all vs filtered for tail, across gammas
if filter_comparison_rows:
    tail_fc = fc_df[fc_df['part'] == 'tail'].sort_values('gamma')

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(tail_fc['gamma'], tail_fc['frac_correct_all'],
            marker='o', color=PART_COLORS['tail'], ls='-', label='all images')
    ax.plot(tail_fc['gamma'], tail_fc['frac_correct_filtered'],
            marker='s', color=PART_COLORS['tail'], ls='--', label=f'px ≥ {MIN_PART_PIXELS}')
    ax.axhline(0.5, color='gray', ls=':', lw=1.5, label='chance')
    ax.set_xlabel('gamma'); ax.set_ylabel('frac_correct')
    ax.set_title(f'Tail z-ordering accuracy: all vs occluded-filtered  (px≥{MIN_PART_PIXELS})')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('fb_mcbm_seg_filter_vs_gamma_tail.png', dpi=150, bbox_inches='tight')
    plt.show()

## 17. Cross-gamma: z_new_orig and z_old_orig distributions

Does the IB penalty (higher γ) suppress donor-concept pre-activation (`z_new_orig`) in original images?  
This is a direct measure of backwash reduction.

In [ ]:
if len(gamma_dfs) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for ax, col, label in [
        (axes[0], 'z_new_orig',
         'z_new_orig per γ  (donor pre-activation; lower = less backwash)'),
        (axes[1], 'z_old_orig',
         'z_old_orig per γ  (source activation; should stay high)'),
    ]:
        tail_vals = {g: df[df['part'] == 'tail'][col].values
                     for g, df in sorted(gamma_dfs.items())}
        bp = ax.boxplot(
            list(tail_vals.values()),
            labels=[f'γ={g}' for g in tail_vals.keys()],
            patch_artist=True,
            medianprops=dict(color='black', lw=2),
        )
        cmap = plt.cm.viridis
        for i, patch in enumerate(bp['boxes']):
            patch.set_facecolor(cmap(i / max(len(tail_vals)-1, 1)))
            patch.set_alpha(0.7)
        ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.7)
        ax.set_ylabel(col + '  (raw z_raw)')
        ax.set_title(label, fontsize=9)
        ax.grid(True, axis='y', alpha=0.3)

    plt.suptitle('Tail: concept pre-activations across γ  (raw MCBM z, no sigmoid)', fontsize=11, y=1.02)
    plt.tight_layout()
    plt.savefig('fb_mcbm_z_orig_across_gammas.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved fb_mcbm_z_orig_across_gammas.png')

## 18. Final summary table

In [ ]:
if gamma_summaries:
    print('=== MCBM Z-ORDERING EXPERIMENT — FULL SUMMARY ===')
    print(f'{"gamma":>8s}  {"part":>6s}  {"frac_correct":>13s}  {"frac_viol":>10s}  '
          f'{"mean_margin":>12s}  {"n_images":>9s}')
    print('-' * 68)
    for g in sorted(gamma_summaries.keys()):
        smry = gamma_summaries[g]
        for _, r in smry.iterrows():
            print(f'{g:>8.1f}  {r["part"]:>6s}  {r["frac_correct"]:>13.3%}  '
                  f'{r["frac_violations"]:>10.3%}  {r["mean_margin"]:>+12.4f}  '
                  f'{int(r["n_images"]):>9d}')
        print()

    print('Interpretation:')
    print('  frac_correct < 100%  → bottleneck not fully visually grounded')
    print('  mean_margin  > 0     → on average correct ordering (even if not always)')
    print('  If frac_correct RISES with γ → IB penalty reduces backwash (desired)')
    print('  If frac_correct FLAT  with γ → backwash is structural, not fixable by IB')